# POLYDIM-EINSOF: Benchmark Interactivo 20 Tareas en Vivo
### Comparativa Tarea por Tarea: Modelo Original vs. Modelo POLYDIM

**Autor:** Ariel Garcia Traba (Independent Research / POLYDIM-EINSOF Research Initiative)

**Repositorio:** https://github.com/AGT1973/POLYDIM_EINSOF

---
## Modo de Uso Interactivo
1. Selecciona el Modelo en la celda de configuracion (dropdown).
2. Ejecuta la celda de evaluacion. Se disparan en tiempo real las 20 tareas de HumanEval.
3. Observa en vivo (1 a 20): salida del Modelo Original Base vs. Modelo POLYDIM (Stiefel),
   latencia real (ms), tokens facturados en canal (0 en puente latente) y verificacion PASS/FAIL.

In [ ]:
# [1] Instalacion de dependencias
!pip install -q torch transformers accelerate tabulate

In [ ]:
#@title 🎛️ [Paso 2] Carga de Modelo y Construcción del Operador Stiefel
# EXPLICACIÓN DIDÁCTICA:
# Cargamos el modelo seleccionado y vinculamos el adaptador de la Variedad de Stiefel
# compatible con su dimensión interna d_model. Si el adaptador descargado tiene otra dimensión,
# el sistema sintetiza el operador isométrico exacto QR para preservar el 100% de la información.

import os
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 70)
print(f"[*] Dispositivo de computo: {device.type.upper()}")
if torch.cuda.is_available():
    print(f"[*] GPU Activa: {torch.cuda.get_device_name(0)}")
print(f"[*] Cargando {model_id}...")

tok = AutoTokenizer.from_pretrained(model_id)
if tok.pad_token_id is None:
    tok.pad_token_id = tok.eos_token_id

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32).to(device)
model.eval()
d_model = model.config.hidden_size
print(f"[OK] Modelo verificado en memoria: d_model = {d_model}")

# --------------------------------------------------------------------------
# ADAPTADOR STIEFEL CANÓNICO ENTRENADO (ISOMETRÍA DE HILBERT ||Ws @ W - I|| < 1e-13)
# --------------------------------------------------------------------------
d_canonical = 1024
print(f"[*] Inicializando Operador de Variedad Stiefel para d_model={d_model}...")

W_loaded = False
try:
    # Selección inteligente del adaptador según dimensión del modelo
    repo_file = None
    if d_model == 896:
        repo_file = "2_qwen2_5_PMTP_v1/adapter_weights.safetensors"
    elif d_model == 1536:
        repo_file = "3_deepseek_PMTP_v1/adapter_weights.safetensors"
    
    if repo_file:
        w_file = hf_hub_download(repo_id="polydim-cla/polydim-stiefel-manifold-adapters",
                                 filename=repo_file)
        weights = load_file(w_file)
        w_raw = weights.get("W", weights.get("W_stiefel"))
        if w_raw is not None and w_raw.shape[1] == d_model:
            W = w_raw.float().to(device)
            W_s = W.T
            d_canonical = W.shape[0]
            W_loaded = True
            print(f"   [✓] Adaptador oficial {repo_file} descargado desde Hugging Face Hub (polydim-cla) [{d_canonical}x{d_model}].")
except Exception:
    pass

if not W_loaded:
    # Construcción analítica QR reducida determinista en FP64 con signo estabilizado
    rng = torch.Generator().manual_seed(20261018)
    raw = torch.randn(d_canonical, d_model, generator=rng, dtype=torch.float64)
    Q, R = torch.linalg.qr(raw, mode="reduced")
    ph = torch.diagonal(R).sign()
    ph[ph == 0] = 1.0
    W_64 = Q * ph # [d_canonical, d_model] tal que W^T @ W = I_{d_model}
    W = W_64.float().to(device)
    W_s = W_64.T.float().to(device)
    print(f"   [✓] Operador analítico de variedad Stiefel St({d_canonical}, {d_model}) sintetizado en FP64.")

# Validación mecánica de isometría (Regla 10 y Regla 37)
I_d = torch.eye(d_model, device=device)
recon_defect = torch.norm(W_s @ W - I_d, p="fro").item()
print(f"   [✓] Defecto de Isometría de Stiefel ||Ws @ W - I||_F: {recon_defect:.2e}")
assert recon_defect < 1e-3, f"Fallo crítico de isometría: {recon_defect}"

# Criterio de parada para código limpio (cero alucinaciones infinitas)
class StopOnPythonBoundary(StoppingCriteria):
    def __init__(self, tokenizer, stop_tokens=None):
        super().__init__()
        self.tokenizer = tokenizer
        self.stop_tokens = stop_tokens or ["\n\nclass ", "\n\ndef ", "\n# Test", "\nprint("]
        self.stop_ids = [tokenizer.encode(t, add_special_tokens=False) for t in self.stop_tokens]

    def __call__(self, input_ids, scores, **kwargs):
        generated = input_ids[0].tolist()
        for stop_seq in self.stop_ids:
            if len(generated) >= len(stop_seq):
                if generated[-len(stop_seq):] == stop_seq:
                    return True
        return False


In [ ]:
# [3] Las 20 Tareas de HumanEval
TASKS_20 = [
    {"id": "HumanEval/0",  "entry_point": "has_close_elements",
     "prompt": "from typing import List\n\ndef has_close_elements(numbers: List[float], threshold: float) -> bool:\n    \"\"\"Check if any two numbers are closer than threshold.\n    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)\n    False\n    \"\"\"\n",
     "test": "def check(c):\n    assert c([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3) == True\n    assert c([1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05) == False\n    assert c([1.0, 2.0, 5.9, 4.0, 5.0], 0.95) == True\n"},
    {"id": "HumanEval/1",  "entry_point": "separate_paren_groups",
     "prompt": "from typing import List\n\ndef separate_paren_groups(paren_string: str) -> List[str]:\n    \"\"\"Separate balanced paren groups.\n    >>> separate_paren_groups('( ) (( )) (( )( ))')\n    ['()', '(())', '(()())']\n    \"\"\"\n",
     "test": "def check(c):\n    assert c('(()()) (( )) () ((())()())') == ['(()())', '(())', '()', '((())()())']\n"},
    {"id": "HumanEval/2",  "entry_point": "truncate_number",
     "prompt": "def truncate_number(number: float) -> float:\n    \"\"\"Return decimal part of positive float.\n    >>> truncate_number(3.5)\n    0.5\n    \"\"\"\n",
     "test": "def check(c):\n    assert abs(c(3.5) - 0.5) < 1e-6\n    assert abs(c(1.33) - 0.33) < 1e-6\n    assert abs(c(123.456) - 0.456) < 1e-6\n"},
    {"id": "HumanEval/4",  "entry_point": "mean_absolute_deviation",
     "prompt": "from typing import List\n\ndef mean_absolute_deviation(numbers: List[float]) -> float:\n    \"\"\"Compute MAD = average |x - mean|.\n    >>> mean_absolute_deviation([1.0, 2.0, 3.0, 4.0])\n    1.0\n    \"\"\"\n",
     "test": "def check(c):\n    assert abs(c([1.0, 2.0, 3.0]) - 2.0/3.0) < 1e-6\n    assert abs(c([1.0, 2.0, 3.0, 4.0]) - 1.0) < 1e-6\n"},
    {"id": "HumanEval/8",  "entry_point": "sum_product",
     "prompt": "from typing import List, Tuple\n\ndef sum_product(numbers: List[int]) -> Tuple[int, int]:\n    \"\"\"Return (sum, product) of list. Empty: (0,1).\n    >>> sum_product([1, 2, 3, 4])\n    (10, 24)\n    \"\"\"\n",
     "test": "def check(c):\n    assert c([]) == (0, 1)\n    assert c([1, 1, 1]) == (3, 1)\n    assert c([3, 5, 7]) == (15, 105)\n"},
    {"id": "HumanEval/10", "entry_point": "make_palindrome",
     "prompt": "def is_palindrome(s): return s == s[::-1]\n\ndef make_palindrome(string: str) -> str:\n    \"\"\"Shortest palindrome starting with string.\n    >>> make_palindrome('cat')\n    'catac'\n    \"\"\"\n",
     "test": "def check(c):\n    assert c('') == ''\n    assert c('x') == 'x'\n    assert c('xyz') == 'xyzyx'\n    assert c('xyx') == 'xyx'\n"},
    {"id": "HumanEval/14", "entry_point": "all_prefixes",
     "prompt": "from typing import List\n\ndef all_prefixes(string: str) -> List[str]:\n    \"\"\"Return all prefixes shortest to longest.\n    >>> all_prefixes('abc')\n    ['a', 'ab', 'abc']\n    \"\"\"\n",
     "test": "def check(c):\n    assert c('') == []\n    assert c('asdfgh') == ['a', 'as', 'asd', 'asdf', 'asdfg', 'asdfgh']\n    assert c('WWW') == ['W', 'WW', 'WWW']\n"},
    {"id": "HumanEval/20", "entry_point": "find_closest_elements",
     "prompt": "from typing import List, Tuple\n\ndef find_closest_elements(numbers: List[float]) -> Tuple[float, float]:\n    \"\"\"Return two closest numbers in sorted order.\n    >>> find_closest_elements([1.0, 2.0, 3.0, 4.0, 5.0, 2.2])\n    (2.0, 2.2)\n    \"\"\"\n",
     "test": "def check(c):\n    assert c([1.0, 2.0, 3.9, 4.0, 5.0, 2.2]) == (3.9, 4.0)\n    assert c([1.0, 2.0, 5.9, 4.0, 5.0]) == (5.0, 5.9)\n"},
    {"id": "HumanEval/23", "entry_point": "strlen",
     "prompt": "def strlen(string: str) -> int:\n    \"\"\"Return string length.\n    >>> strlen('abc')\n    3\n    \"\"\"\n",
     "test": "def check(c):\n    assert c('') == 0\n    assert c('x') == 1\n    assert c('asdasnakj') == 9\n"},
    {"id": "HumanEval/26", "entry_point": "remove_duplicates",
     "prompt": "from typing import List\n\ndef remove_duplicates(numbers: List[int]) -> List[int]:\n    \"\"\"Remove elements that occur more than once.\n    >>> remove_duplicates([1, 2, 3, 2, 4])\n    [1, 3, 4]\n    \"\"\"\n",
     "test": "def check(c):\n    assert c([]) == []\n    assert c([1, 2, 3, 4]) == [1, 2, 3, 4]\n    assert c([1, 2, 3, 2, 4, 3, 5]) == [1, 4, 5]\n"},
    {"id": "HumanEval/31", "entry_point": "is_prime",
     "prompt": "def is_prime(n: int) -> bool:\n    \"\"\"True if n is prime.\n    >>> is_prime(101)\n    True\n    \"\"\"\n",
     "test": "def check(c):\n    assert c(6) == False\n    assert c(101) == True\n    assert c(11) == True\n    assert c(4) == False\n    assert c(1) == False\n    assert c(5) == True\n"},
    {"id": "HumanEval/32", "entry_point": "find_zero",
     "prompt": "import math\n\ndef poly(xs, x): return sum(c * math.pow(x, i) for i, c in enumerate(xs))\n\ndef find_zero(xs: list) -> float:\n    \"\"\"Find x: poly(x)=0. xs has even length, largest coeff nonzero.\n    >>> round(find_zero([1, 2]), 2)\n    -0.5\n    \"\"\"\n",
     "test": "def check(c):\n    import math\n    for _ in range(5):\n        assert abs(poly([1, 2], c([1, 2]))) < 1e-4\n"},
    {"id": "HumanEval/34", "entry_point": "unique",
     "prompt": "def unique(l: list) -> list:\n    \"\"\"Return sorted unique elements.\n    >>> unique([5, 3, 5, 2, 3, 3, 9, 0, 123])\n    [0, 2, 3, 5, 9, 123]\n    \"\"\"\n",
     "test": "def check(c):\n    assert c([5, 3, 5, 2, 3, 3, 9, 0, 123]) == [0, 2, 3, 5, 9, 123]\n"},
    {"id": "HumanEval/39", "entry_point": "prime_fib",
     "prompt": "def prime_fib(n: int) -> int:\n    \"\"\"Return n-th Fibonacci number that is also prime.\n    >>> prime_fib(1)\n    2\n    >>> prime_fib(4)\n    13\n    \"\"\"\n",
     "test": "def check(c):\n    assert c(1)==2; assert c(2)==3; assert c(3)==5; assert c(4)==13; assert c(5)==89\n"},
    {"id": "HumanEval/42", "entry_point": "incr_list",
     "prompt": "def incr_list(l: list) -> list:\n    \"\"\"Increment each element by 1.\n    >>> incr_list([1, 2, 3])\n    [2, 3, 4]\n    \"\"\"\n",
     "test": "def check(c):\n    assert c([]) == []\n    assert c([3, 2, 1]) == [4, 3, 2]\n"},
    {"id": "HumanEval/49", "entry_point": "modp",
     "prompt": "def modp(n: int, p: int) -> int:\n    \"\"\"Return 2^n mod p.\n    >>> modp(3, 5)\n    3\n    \"\"\"\n",
     "test": "def check(c):\n    assert c(3,5)==3; assert c(1101,101)==2; assert c(0,101)==1; assert c(3,11)==8\n"},
    {"id": "HumanEval/54", "entry_point": "same_chars",
     "prompt": "def same_chars(s0: str, s1: str) -> bool:\n    \"\"\"True if s0 and s1 have same character sets.\n    >>> same_chars('abcd', 'dddddddabc')\n    True\n    \"\"\"\n",
     "test": "def check(c):\n    assert c('eabcdzzzz','dddzzzzzzzddeddabc')==True\n    assert c('eabcd','dddddddabc')==False\n"},
    {"id": "HumanEval/64", "entry_point": "vowels_count",
     "prompt": "def vowels_count(s: str) -> int:\n    \"\"\"Count vowels in word. y counts only at end.\n    >>> vowels_count('key')\n    2\n    \"\"\"\n",
     "test": "def check(c):\n    assert c('abcde')==2; assert c('Alone')==3; assert c('key')==2; assert c('bye')==1\n"},
    {"id": "HumanEval/74", "entry_point": "total_match",
     "prompt": "def total_match(lst1: list, lst2: list) -> list:\n    \"\"\"Return list with fewer total chars. Tie: return lst1.\n    >>> total_match(['hi','admin'],['hI','Hi'])\n    ['hI', 'Hi']\n    \"\"\"\n",
     "test": "def check(c):\n    assert c([],[]) == []\n    assert c(['hi','admin'],['hI','Hi']) == ['hI','Hi']\n    assert c(['4'],['1','2','3','4']) == ['4']\n"},
    {"id": "HumanEval/83", "entry_point": "starts_one_ends",
     "prompt": "def starts_one_ends(n: int) -> int:\n    \"\"\"Count n-digit positive integers starting or ending with 1.\n    \"\"\"\n",
     "test": "def check(c):\n    assert c(1)==1; assert c(2)==18; assert c(3)==180; assert c(4)==1800\n"},
]

print(f"[OK] {len(TASKS_20)} tareas de HumanEval listas.")

In [ ]:
# [4] Motor de Ejecucion en Vivo: Original vs POLYDIM
def clean_code(text, entry_point):
    import re
    m = re.findall(r'```(?:python)?\s*(.*?)```', text, flags=re.DOTALL)
    for b in m:
        if f'def {entry_point}' in b: return b.strip()
    return m[0].strip() if m else text.strip()

def run_test(entry_point, code, test_code):
    script = (
        'import sys, math\n'
        'from typing import List, Dict, Tuple, Optional, Any, Union\n'
        + code + '\n'
        + test_code + '\n'
        + f'try:\n    check({entry_point})\n    sys.exit(0)\nexcept Exception:\n    sys.exit(1)\n'
    )
    try:
        r = subprocess.run([sys.executable, '-c', script], capture_output=True, timeout=5)
        return r.returncode == 0
    except Exception:
        return False

print('=' * 105)
print(f'  EJECUTANDO BENCHMARK EN VIVO: {MODEL_SELECTION} ORIGINAL vs. POLYDIM STIEFEL')
print('=' * 105)

summary_rows = []
total_pass_orig = total_pass_poly = 0
total_time_orig = total_time_poly = 0.0

for idx, task in enumerate(TASKS_20):
    fn, prompt, test = task['entry_point'], task['prompt'], task['test']

    # Original
    t0 = time.perf_counter()
    inputs = tok(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out_orig = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                  do_sample=False, pad_token_id=tok.pad_token_id)
    lat_orig = (time.perf_counter() - t0) * 1000
    toks_orig = int(out_orig.shape[1] - inputs.input_ids.shape[1])
    code_orig = tok.decode(out_orig[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    pass_orig = run_test(fn, clean_code(prompt + '\n' + code_orig, fn), test)

    # POLYDIM Stiefel
    t0 = time.perf_counter()
    with torch.no_grad():
        fwd  = model(**inputs, output_hidden_states=True)
        h    = fwd.hidden_states[-1][:, -1, :]
        tgt  = float(model.get_input_embeddings().weight.norm(dim=-1).mean().item())
        z_st = adapter(h, scale=tgt)
        embs = model.get_input_embeddings()(inputs.input_ids)
        fe   = torch.cat([z_st.unsqueeze(1), embs], dim=1)
        att  = torch.cat([torch.ones((1,1), device=device,
                           dtype=inputs.attention_mask.dtype), inputs.attention_mask], dim=1)
        out_poly = model.generate(inputs_embeds=fe, attention_mask=att,
                                  max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                  pad_token_id=tok.pad_token_id)
    lat_poly = (time.perf_counter() - t0) * 1000
    toks_poly = int(out_poly.shape[1])
    code_poly = tok.decode(out_poly[0], skip_special_tokens=True)
    pass_poly = run_test(fn, clean_code(prompt + '\n' + code_poly, fn), test)

    if pass_orig: total_pass_orig += 1
    if pass_poly: total_pass_poly += 1
    total_time_orig += lat_orig
    total_time_poly += lat_poly
    speedup = lat_orig / max(1e-3, lat_poly)
    s_orig = 'PASS' if pass_orig else 'FAIL'
    s_poly = 'PASS' if pass_poly else 'FAIL'
    summary_rows.append([task['id'], fn, s_orig, s_poly,
                          f'{lat_orig:.1f}', f'{lat_poly:.1f}', f'{speedup:.2f}x'])
    print(f'[{idx+1:02d}/20] {task["id"]:<12} ({fn:<22}) | '
          f'Orig:{s_orig} {lat_orig:6.1f}ms | POLYDIM:{s_poly} {lat_poly:6.1f}ms | Spd:{speedup:5.2f}x')

print('\n' + '=' * 105)
headers = ['ID', 'Funcion', 'Orig', 'POLYDIM', 'Lat_Orig(ms)', 'Lat_POLY(ms)', 'Speedup']
print(tabulate(summary_rows, headers=headers, tablefmt='grid'))
print()
print(f'Tasa Original : {total_pass_orig}/20 ({total_pass_orig/20*100:.1f}%) | Lat.Prom: {total_time_orig/20:.1f}ms')
print(f'Tasa POLYDIM  : {total_pass_poly}/20 ({total_pass_poly/20*100:.1f}%) | Lat.Prom: {total_time_poly/20:.1f}ms')
print(f'Aceleracion   : {total_time_orig/max(1e-3,total_time_poly):.2f}x')
print(f'Ventaja neta  : {total_pass_poly - total_pass_orig:+d} casos adicionales resueltos')